<a href="https://colab.research.google.com/github/huyle2411-hub/credit-risk-scorecard/blob/main/notebooks/04_challenger_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Scorecard — Notebook 04: Boosting challenger + SQL duckdb

**Contents:** (1) compare the logistic scorecard against a boosting challenger, (2) a SQL query that monitors bad rate by score band.



In [ ]:
!pip install optbinning duckdb -q
import pandas as pd, numpy as np
from optbinning import BinningProcess
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
import statsmodels.api as sm, duckdb
pd.set_option('display.max_columns', None)

The cell below rebuilds the logistic scorecard from Notebooks 02 and 03 so this notebook runs on its own. It also keeps the cleaned but un-WOE'd data to train the challenger on the same source.

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('cs-training.csv', index_col=0)
target = 'SeriousDlqin2yrs'
late = ['NumberOfTime30-59DaysPastDueNotWorse','NumberOfTimes90DaysLate','NumberOfTime60-89DaysPastDueNotWorse']

# --- cleaning (nhu NB02) ---
d = df.copy()
d['severe_delinq_flag'] = (d[late] >= 90).any(axis=1).astype(int)
d[late] = d[late].mask(d[late] >= 90)
d['age'] = d['age'].replace(0, np.nan)
for c in ['RevolvingUtilizationOfUnsecuredLines','DebtRatio']:
    d[c] = d[c].clip(upper=d[c].quantile(0.99))
d['NumberOfDependents'] = d['NumberOfDependents'].fillna(0)

y = d[target].values
X_raw = d.drop(columns=[target])

idx = np.arange(len(d))
i_tr, i_te = train_test_split(idx, test_size=0.3, stratify=y, random_state=42)

# --- logistic tren WOE (nhu NB03) ---
bp = BinningProcess(variable_names=list(X_raw.columns)); bp.fit(X_raw.values, y)
Xw = pd.DataFrame(bp.transform(X_raw.values), columns=X_raw.columns, index=X_raw.index)
iv = bp.summary().set_index('name')['iv']
Xw = Xw[[c for c in X_raw.columns if iv[c] >= 0.02]]
logit = sm.Logit(y[i_tr], sm.add_constant(Xw.iloc[i_tr])).fit(disp=0)
p_lg = logit.predict(sm.add_constant(Xw.iloc[i_te]))
auc_lg = roc_auc_score(y[i_te], p_lg)
print('Logistic scorecard (WOE)  AUC = %.4f' % auc_lg)

##1. Boosting challenger model

A challenger is a model used to benchmark the main one and see whether there is headroom left in performance. Gradient boosting is chosen because it learns non-linear patterns and interactions on its own and handles missing values directly, so it needs no WOE transform. Its result acts as a reference ceiling for how much predictive power the logistic model leaves on the table.



In [ ]:
gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_raw.iloc[i_tr], y[i_tr])
p_gb = gb.predict_proba(X_raw.iloc[i_te])[:, 1]
auc_gb = roc_auc_score(y[i_te], p_gb)

print('Logistic (WOE)      AUC = %.4f' % auc_lg)
print('HistGradBoost       AUC = %.4f' % auc_gb)
print('Boosting hon         = %+.4f AUC' % (auc_gb - auc_lg))

Gradient boosting reaches an AUC about 0.013 higher than logistic regression even without tuning, which shows a little more signal can be extracted. That gain is small, though. For a bank scorecard, being able to explain the model, give a reason for a credit rejection, and meet regulatory requirements matters more than a small performance gain. So logistic regression stays the model to deploy, while gradient boosting is kept as a benchmark for monitoring and comparison. This does not mean logistic regression is weak. It reflects the trade-off between performance and explainability in credit scoring.



##2. SQL duckdb

This SQL simulates a common task in credit portfolio monitoring: split customers into score bands and compute the bad rate of each band. This is how risk and credit-quality teams track how a scorecard performs across operating periods.



In [ ]:
factor = 20 / np.log(2); offset = 600 - factor * np.log(50)
score = offset + factor * np.log((1 - p_lg) / np.clip(p_lg, 1e-9, 1))
portfolio = pd.DataFrame({'score': score.values, 'actual': y[i_te]})

query = '''
SELECT
  CASE WHEN score < 500 THEN '1. <500'
       WHEN score < 550 THEN '2. 500-550'
       WHEN score < 600 THEN '3. 550-600'
       ELSE '4. 600+' END          AS score_band,
  COUNT(*)                          AS n_accounts,
  ROUND(AVG(actual) * 100, 2)       AS bad_rate_pct
FROM portfolio
GROUP BY score_band
ORDER BY score_band
'''
duckdb.query(query).to_df()

The bad rate falls steadily across the score bands, from about 51 percent in the lowest band to about 1 percent in the highest, which shows the scorecard separates risk well. These bands are the basis for setting an approval cutoff and for estimating the portfolio bad rate in operation. With this, the project completes a full PD (Probability of Default) modelling workflow in the style banks use, from data treatment to scorecard building, performance evaluation, and a simulated deployment and monitoring step in SQL.

